In [ ]:
# 1. Google Drive 마운트
from google.colab import drive
import os
import zipfile
import unicodedata

# Drive 마운트 (승인 팝업 필요)
drive.mount('/content/drive')

# 2. 기본 경로 및 디렉토리 구조 설정
BASE_PATH = '/content/drive/MyDrive/5stone_experiment/1_clustering_test'
DIR_STRUCTURE = [
    'data/raw',
    'data/processed',
    'embeddings',
    'vector_db',
    'checkpoints',
    'config'
]

print("디렉토리 구조를 생성합니다...")
for dir_name in DIR_STRUCTURE:
    dir_path = os.path.join(BASE_PATH, dir_name)
    os.makedirs(dir_path, exist_ok=True)
    print(f"생성 완료: {dir_path}")

# 3. .env 파일 생성 (존재하지 않을 경우만)
env_path = os.path.join(BASE_PATH, 'config', '.env')
if not os.path.exists(env_path):
    with open(env_path, 'w') as f:
        f.write("OPENAI_API_KEY=your_openai_api_key_here\n")
        f.write("HF_TOKEN=your_huggingface_token_here\n")
    print(f".env 파일 생성 완료: {env_path}")
else:
    print(f".env 파일이 이미 존재합니다: {env_path}")

# 4. macOS 호환성을 고려한 dataset.zip 안전 압축 해제
zip_path = os.path.join(BASE_PATH, 'dataset.zip')
extract_path = os.path.join(BASE_PATH, 'data/raw')

print(f"\n압축 해제를 시작합니다: {zip_path}")
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for member in zip_ref.infolist():
            # 1단계: macOS zip 인코딩 오류 우회 (cp437 -> utf-8 변환)
            try:
                name = member.filename.encode('cp437').decode('utf-8')
            except UnicodeDecodeError:
                name = member.filename.encode('cp437').decode('euc-kr', 'ignore')

            # 2단계: NFD(macOS) -> NFC(Linux/Windows) 유니코드 정규화
            name = unicodedata.normalize('NFC', name)

            # 3단계: macOS 시스템 파일 및 불필요한 메타데이터 필터링
            if '__MACOSX' in name or '.DS_Store' in name:
                continue

            # 대상 경로 설정 및 해제
            target_path = os.path.join(extract_path, name)

            if member.is_dir():
                os.makedirs(target_path, exist_ok=True)
            else:
                os.makedirs(os.path.dirname(target_path), exist_ok=True)
                with open(target_path, 'wb') as outfile:
                    outfile.write(zip_ref.read(member))
    print(f"압축 해제 및 정규화 완료: {extract_path}")
else:
    print(f"오류: {zip_path} 파일을 찾을 수 없습니다.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
디렉토리 구조를 생성합니다...
생성 완료: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/raw
생성 완료: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/processed
생성 완료: /content/drive/MyDrive/5stone_experiment/1_clustering_test/embeddings
생성 완료: /content/drive/MyDrive/5stone_experiment/1_clustering_test/vector_db
생성 완료: /content/drive/MyDrive/5stone_experiment/1_clustering_test/checkpoints
생성 완료: /content/drive/MyDrive/5stone_experiment/1_clustering_test/config
.env 파일 생성 완료: /content/drive/MyDrive/5stone_experiment/1_clustering_test/config/.env

압축 해제를 시작합니다: /content/drive/MyDrive/5stone_experiment/1_clustering_test/dataset.zip


In [ ]:
from google.colab import drive
drive.flush_and_unmount()

In [ ]:
from google.colab import runtime
runtime.unassign()